You can enter resources you have access to for comparison.

In [9]:
setups = [
    {"name": "RTX 4090 (vast.ai)", "tflops": 81.4, "cost_hr": 0.338},
    {"name": "8x RTX 4090 (vast.ai)", "tflops": 651.2, "cost_hr": 2.889},
    {"name": "2x RTX 4070S Ti (vast.ai)", "tflops": 84.8, "cost_hr": 0.362},
    {"name": "2x RTX 5070 Ti (vast.ai)", "tflops": 88.6, "cost_hr": 0.338}
]

Architecture variables. Copy from GPTConfig in model.py before playing with this notebook to make sure they are up to date. If you update the architecture you will need to change how the calculations are done. If you are only playing with hyperparamets you shouldn't need to though.

In [10]:
from model import LLM, LLMConfig

# enter architecture variables here
depth = 12
block_size = 1024
vocab_size = 50304 # using the one in train.py here instead of model.py

mfu = 1
total_batch_size = 524288

model = LLM(LLMConfig(depth=depth, block_size=block_size, vocab_size=vocab_size))
total_params = sum(p.numel() for p in model.parameters())
print(f"total parameters {total_params//1000000}M")

total parameters 142M


Apply Chinchilla Scaling Laws. Chinchilla paper suggests 20 tokens per parameter for compute-optimal training.

In [11]:
# this cell has manually put in and adjusted weights in order to calculate predictions similiar to my past experiences

def calc_metrics(params, tokens, tflops, mfu, cost_hr):
    total_flops = 6 * params * tokens
    achieved_flops_per_sec = (tflops * 1000000000000) * mfu
    hours = (total_flops / achieved_flops_per_sec) / 3600
    cost = hours * cost_hr
    return hours/1.5, cost

chinchilla_tokens = 20 * total_params
overtrain_5B = 5000000000
overtrain_10B = 10000000000

for setup in setups:
    h_chin, c_chin = calc_metrics(total_params, chinchilla_tokens, setup['tflops'], mfu, setup['cost_hr'])
    h_5b, c_5b = calc_metrics(total_params, overtrain_5B, setup['tflops'], mfu, setup['cost_hr'])
    h_10b, c_10b = calc_metrics(total_params, overtrain_10B, setup['tflops'], mfu, setup['cost_hr'])
    
    print(f"\n[{setup['name']}]")
    print(f"  Chinchilla (2.5B): {h_chin:.1f} hrs | ${c_chin:.2f} | steps {chinchilla_tokens//total_batch_size}")
    print(f"  Overtrain  (5.0B): {h_5b:.1f} hrs | ${c_5b:.2f} | steps {overtrain_5B//total_batch_size}")
    print(f"  Heavy      (10B):  {h_10b:.1f} hrs | ${c_10b:.2f} | steps {overtrain_10B//total_batch_size}")


[RTX 4090 (vast.ai)]
  Chinchilla (2.5B): 5.5 hrs | $2.81 | steps 5435
  Overtrain  (5.0B): 9.7 hrs | $4.93 | steps 9536
  Heavy      (10B):  19.4 hrs | $9.86 | steps 19073

[8x RTX 4090 (vast.ai)]
  Chinchilla (2.5B): 0.7 hrs | $3.00 | steps 5435
  Overtrain  (5.0B): 1.2 hrs | $5.27 | steps 9536
  Heavy      (10B):  2.4 hrs | $10.54 | steps 19073

[2x RTX 4070S Ti (vast.ai)]
  Chinchilla (2.5B): 5.3 hrs | $2.89 | steps 5435
  Overtrain  (5.0B): 9.3 hrs | $5.07 | steps 9536
  Heavy      (10B):  18.7 hrs | $10.14 | steps 19073

[2x RTX 5070 Ti (vast.ai)]
  Chinchilla (2.5B): 5.1 hrs | $2.58 | steps 5435
  Overtrain  (5.0B): 8.9 hrs | $4.53 | steps 9536
  Heavy      (10B):  17.9 hrs | $9.06 | steps 19073
